In [1]:
import numpy as np 
import pandas as pd
import re 
import datetime
import traceback as tb
import sys
import math
import unicodedata
from datetime import time

# Limpieza de datos de CTD 2023

In [13]:
df_ctd = pd.read_excel(r'C:\Users\felipe.abarzua\Desktop\workspace\DATA_AMBIENTALES\Proyecto Seguimiento Ambiental\CTD_2019-2024\5_BD_CTD_SEGUIMIENTO_2023.xlsx')

In [15]:
df_ctd

,Nombre del Proyecto:,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICULTURA EN CHILE Y SU EFECTO EN LOS ECOSISTEMAS DE EMPLAZAMIENTO 2023 -2024,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26
0,Código del Proyecto:,656-160,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Etapa Proyecto:,Objetivo 1. Actividad 2. Muestreos y Análisis ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Jefe Proyecto,Johana Ojeda,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Institución:,Instituto de Fomento Pesquero,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Sectores:,"Estuario de Reloncaví, Seno de Reloncaví, Golf...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6110,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,656-160,13-5-2023,13-5-2023,CTD SeaBird SBE 19plus V 2.5.2 SERIAL NO. 756...,-45.668,-73.299,32,249,253,...,23.1065,NaN,NaN,NaN,X,NaN,32,FIORDO QUITRALCO,13:04:00,1.5
6111,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,656-160,13-5-2023,13-5-2023,CTD SeaBird SBE 19plus V 2.5.2 SERIAL NO. 756...,-45.668,-73.299,32,250,253,...,23.1066,NaN,NaN,NaN,X,NaN,32,FIORDO QUITRALCO,13:04:00,1.5
6112,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,656-160,13-5-2023,13-5-2023,CTD SeaBird SBE 19plus V 2.5.2 SERIAL NO. 756...,-45.668,-73.299,32,251,253,...,23.1066,NaN,NaN,NaN,X,NaN,32,FIORDO QUITRALCO,13:04:00,1.5
6113,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,656-160,13-5-2023,13-5-2023,CTD SeaBird SBE 19plus V 2.5.2 SERIAL NO. 756...,-45.668,-73.299,32,252,253,...,23.1071,NaN,NaN,NaN,X,NaN,32,FIORDO QUITRALCO,13:04:00,1.5


In [88]:
def limpieza_ctd(df):

    # Creamos una copia del dataset
    df_ctd_copy = df.copy()

    # Eliminamos las filas innecesarias
    df_ctd_copy = df_ctd_copy.iloc[19:  , : ]

    # Dejamos la primera fila como columnas
    df_ctd_copy.columns = df_ctd_copy.iloc[0]

    # Eliminamos la primera fila 
    df_ctd_copy = df_ctd_copy.iloc[1: , :]

    #Reiniciamos los indices
    df_ctd_copy = df_ctd_copy.reset_index(drop = True)

    #Agregamos la columna ID 
    df_ctd_copy.insert(0 , 'ID' , df_ctd_copy.index + 1)

    # Modificamos las columnas de estación ya que existen dos de ellas
    df_ctd_copy.columns.values[8] = 'ESTACION_1'
    df_ctd_copy.columns.values[24] = 'ESTACION_2'

    #Creamos la lista de columnas 
    columnas = df_ctd_copy.columns.to_list()

    #Eliminamos la estacion_2 ya que es innecesaria
    df_ctd_copy.drop(columns=['ESTACION_2'], inplace=True)

    # Renombramos la ESTACION_1 por ESTACION
    df_ctd_copy.columns.values[8] = 'ESTACION'

    #Creamos la lista de columnas 
    columnas = df_ctd_copy.columns.to_list()
    
    #Creamos las columnas fijas y variables
    columnas_fijas = ['ID', 'NOMB_PROY', 'COD_PROY', 'FECHA_INI', 'FECHA_TER', 'EQUIPO', 'LATITUD', 'LONGUITUD', 'ESTACION', 'PROF_EQ', 'PROF_SECT' , 'HORA_INICIO' , 'COD_REG' , 'SECTOR']
    columnas_variables = [ 'TEMPERATUR', 'OXIG_ml/L', 'OXIG_mg/L', 'OXIG_%sat', 'OX_umol/kg','SALINIDAD', 'DENSIDAD', 'CLOROFILA', 'FEOPIGMEN' , 'D_SECCHI']

    
    #Realizamos un melt para convertir las columnas_variables en filas 
    df_ctd_copy = pd.melt(df_ctd_copy , id_vars= columnas_fijas , value_vars= columnas_variables , var_name= 'VARIABLE' , value_name='VALOR')

    #Vamos a reemplazar los valores e HORA_INICIO ya que están mal escritos

    def normalizar_hora_string(hora_str):

            #Normaliza una cadena de tiempo para asegurar que los minutos tengan dos dígitos.
            #Ej: '12:5' se convierte en '12:05'.
            
        if pd.isna(hora_str): # Maneja posibles valores NaN/nulos si los hubiera
            return hora_str

        partes = str(hora_str).split(':')
        if len(partes) != 2:
                # Manejar casos donde el formato no es 'HH:MM' (ej. ya está mal, o es un dato inesperado)
                # Puedes decidir si quieres levantar un error, devolver el original, o un valor específico.
                # Por simplicidad, devolveremos el original si el formato no es el esperado de dos partes.
            return hora_str

        horas = partes[0]
        minutos = partes[1]

        if len(minutos) == 1:
            minutos = '0' + minutos # Añadir el cero delante si es un solo dígito

            return f"{horas}:{minutos}"
        
        #Aplicamos la funcion de hora a la columnas HORA_INICIO
        df_ctd_copy['HORA_INICIO'] = df_ctd_copy['HORA_INICIO'].apply(normalizar_hora_string)

        # Creamos una funcion para convertir la columna de HORA_INICIO a datetime para luego Crear una columna de fecha y hora 
    def convertir_hora(time_value):

        if isinstance(time_value, str): 
                
                # Separamos el ":" del texto y lo convertimos a entero y obtenemos dos variables
                horas, minutos = map(int, time_value.split(':'))
                #Retornamos los valores de Horas y minutos
                return datetime.time(horas, minutos)
        else:
                # En caso que ya es datetime.time se deja como esta
            return time_value


    
    df_ctd_copy['HORA_INICIO'] = df_ctd_copy['HORA_INICIO'].apply(convertir_hora)


    #FECHA INI CONVERTIDA A STRING 
    df_ctd_copy['FECHA_INI_STR'] =  pd.to_datetime(df_ctd_copy['FECHA_INI'] , format = '%d-%m-%Y' , dayfirst= True).dt.strftime("%d/%m/%Y")
    df_ctd_copy['FECHA_TERMINO'] = pd.to_datetime(df_ctd_copy['FECHA_TER'] , format = '%d-%m-%Y' , dayfirst= True).dt.strftime("%d/%m/%Y")

     
        #Creamos una funcion lambda para combinar FECHA_INI con la HORA_INI
    df_ctd_copy['FECHA_INICIO'] = df_ctd_copy.apply(
        lambda row: datetime.datetime.strptime(row['FECHA_INI_STR'], "%d/%m/%Y").replace(
            hour=row['HORA_INICIO'].hour,
            minute=row['HORA_INICIO'].minute,
            second=row['HORA_INICIO'].second
            ),
            axis=1
        )

        # Modificamos el formato de FECHA_INICIO 
    df_ctd_copy['FECHA_INICIO'] = df_ctd_copy['FECHA_INICIO'].dt.strftime("%d/%m/%Y %H:%M:%S")

        # Eliminamos las columnas innecesarias

    df_ctd_copy = df_ctd_copy[['ID', 'NOMB_PROY', 'COD_PROY', 'EQUIPO','LATITUD', 'LONGUITUD', 'ESTACION', 'PROF_EQ', 'PROF_SECT', 'COD_REG', 'SECTOR', 'VARIABLE', 'VALOR',
                                'FECHA_TERMINO', 'FECHA_INICIO']]

        # Cambiamos las variables
    df_ctd_copy['VARIABLE'] = np.where(df_ctd_copy['VARIABLE'] == 'D_SECCHI' , 'DISCO SECCHI' , df_ctd_copy['VARIABLE'] )
    df_ctd_copy['VARIABLE'] = np.where(df_ctd_copy['VARIABLE'] == 'TEMPERATUR' , 'TEMPERATURA' , df_ctd_copy['VARIABLE'] )
        
        # Dejamos las variable en mayuscula
    df_ctd_copy['VARIABLE'] = df_ctd_copy['VARIABLE'].str.upper()

        # Modificamos los valores

    df_ctd_copy['VALOR'] = np.where(df_ctd_copy['VALOR'] == 'sd' , np.nan , df_ctd_copy['VALOR'])

    #Renombramos las columnas 

    df_ctd_copy = df_ctd_copy.rename(columns={'PROF_EQ' : 'PROF_EQU' , 'LONGUITUD' : 'LONGITUD'})

    #Renombramos la columna de NOMB_PROY

    df_ctd_copy['NOMB_PROY'] = 'ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICULTURA EN CHILE Y SU EFECTO EN LOS ECOSISTEMAS DE EMPLAZAMIENTO 2023-2024'
                
    #Renombramos los Equipos

    df_ctd_copy['EQUIPO'] = 'CTD SeaBird SBE 19plus SERIAL NO.7566'


    return df_ctd_copy , columnas



In [89]:
df_ctd_copy , columnas= limpieza_ctd(df_ctd)

In [87]:
df_ctd_copy['EQUIPO'].unique()

array(['CTD SeaBird SBE 19plus V 2.5.2  SERIAL NO. 7566// SBE19plusV2_7566.xmlcon'],
      dtype=object)

In [86]:
print(df_ctd_copy['VALOR'].unique().tolist())

[8.2827, 8.4874, 8.7781, 9.4504, 9.833, 10.3585, 10.8388, 10.9111, 10.9167, 10.9518, 10.8877, 10.8535, 10.8506, 10.8611, 10.8976, 10.915, 10.929, 10.9573, 10.9759, 10.9917, 11.0137, 11.0416, 11.0585, 11.0454, 11.0262, 11.0166, 11.0072, 11.002, 10.9902, 10.9818, 10.9688, 10.9618, 10.9602, 10.9592, 10.9582, 10.958, 10.9579, 10.9587, 10.9585, 10.9581, 10.9566, 10.9558, 10.9555, 10.9556, 10.9552, 10.9547, 10.9544, 10.9542, 10.9536, 10.9533, 10.9532, 10.9527, 10.9525, 10.9524, 10.9522, 10.9515, 10.951, 10.9509, 10.9508, 10.9506, 10.9502, 10.9501, 10.95, 10.9496, 10.9493, 10.9494, 10.9495, 10.9491, 10.9488, 10.9487, 10.9484, 10.9483, 10.948, 10.9478, 10.9481, 10.9482, 10.9486, 10.9499, 10.9511, 10.9519, 10.9523, 10.9529, 10.9535, 10.9537, 10.9539, 10.9541, 10.9545, 10.9548, 10.9551, 10.9554, 10.9561, 10.9567, 10.9571, 10.9575, 10.9589, 10.9595, 10.9597, 10.9601, 10.9604, 10.9605, 10.9609, 10.9606, 8.9275, 8.7787, 8.7468, 8.7584, 8.848, 9.4689, 10.06, 10.3924, 10.577, 10.6522, 10.6897, 10.699

In [90]:
df_ctd_copy.to_csv('output/2023/AMBIENTALES_CTD_2023.csv' , index= False)

# Limpieza de datos Fisico Quimico 2023

In [92]:
df_fisico = pd.read_excel(r'C:\Users\felipe.abarzua\Desktop\workspace\DATA_AMBIENTALES\Proyecto Seguimiento Ambiental\FISICO QUIMICOS_2012-2024\1_BD_FISICOQUIMICO_2023.xlsx')

In [94]:
df_fisico

,Nombre del Proyecto:,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICULTURA EN CHILE Y SU EFECTO EN LOS ECOSISTEMAS DE EMPLAZAMIENTO 2023-2024,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,Etapa Proyecto:,Objetivo 1. Actividad 2. Muestreos y Analisis ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Jefe Proyecto,Gastón Vidal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Institucion:,INSTITUTO DE FOMENTO PESQUERO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Nombre Sector:,"Coquimbo (Tongoy), Seno Reloncaví, Estuario d...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Región:,"Región de Coquimbo, Región de Los Lagos, Regi...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Gastón Vidal,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2023.Proy.SeguimientoAmbiental,Natales,9,"51°46'01,6''","72°54'05,0''",7.953333,...,7.266667,0.11547,162.633333,25.971972,26.98233,17.723982,1.961747,2023-08-27 00:00:00,12:32:00,9
159,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Gastón Vidal,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2023.Proy.SeguimientoAmbiental,Natales,10,"52°03'57,8''","72°56'02,0''",6.566667,...,7.166667,0.152753,291.3,30.677027,86.953482,62.591727,3.1514,2023-08-28 00:00:00,12:30:00,10
160,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Gastón Vidal,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2023.Proy.SeguimientoAmbiental,Natales,11,"52°11'59,7''","72°57'03,6''",7.396667,...,7.333333,0.057735,185.333333,29.309441,71.829152,28.643757,12.449799,2023-08-27 00:00:00,16:54:00,11
161,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Gastón Vidal,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2023.Proy.SeguimientoAmbiental,Natales,13,"52°13'10,4''","72°52'26,1''",7.143333,...,7.166667,0.152753,277.466667,4.56545,0.012654,0.021894,2.819486,2023-08-27 00:00:00,17:30:00,13


In [96]:
df_fisico_copy

14,ID,NOMB_PROY,ETAP_PROY,JEFE_PROY,INST_PROY,CAMPAÑA,ZONA,ESTACION,Latitud S,Longitud W,pH,Temperatura (°C),Redox EHNHE (mV),Sulfuros (µM),Materia orgánica total (%),Fecha,Hora Muestreo
0,1,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Gastón Vidal,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2023.Proy.SeguimientoAmbiental,Estuario,1,"41°26'14,9''","72°17'47,9''",7.023333,11.366667,116.733333,51.551157,4.958564,11/06/2023,09:04:00
1,2,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Gastón Vidal,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2023.Proy.SeguimientoAmbiental,Estuario,2,"41°28'07,3''","72°18'43,5''",7.263333,11.466667,103.566667,3.088821,3.357422,11/06/2023,08:40:00
2,3,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Gastón Vidal,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2023.Proy.SeguimientoAmbiental,Estuario,3,"41°29'45,4''","72°18'45,5''",7.326667,11.233333,154.833333,17.46407,2.099861,11/06/2023,09:35:00
3,4,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Gastón Vidal,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2023.Proy.SeguimientoAmbiental,Estuario,5,"41°30'39,3''","72°17'21,1''",6.926667,11.1,266.533333,56.783292,2.271398,11/06/2023,10:10:00
4,5,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Gastón Vidal,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2023.Proy.SeguimientoAmbiental,Estuario,6,"41°31'32,6''","72°20'21,4''",7.316667,11.5,167.133333,23.691977,4.9482,10/06/2023,15:40:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143,144,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Gastón Vidal,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2023.Proy.SeguimientoAmbiental,Natales,9,"51°46'01,6''","72°54'05,0''",7.953333,7.266667,162.633333,26.98233,1.961747,2023-08-27 00:00:00,12:32:00
144,145,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Gastón Vidal,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2023.Proy.SeguimientoAmbiental,Natales,10,"52°03'57,8''","72°56'02,0''",6.566667,7.166667,291.3,86.953482,3.1514,2023-08-28 00:00:00,12:30:00
145,146,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Gastón Vidal,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2023.Proy.SeguimientoAmbiental,Natales,11,"52°11'59,7''","72°57'03,6''",7.396667,7.333333,185.333333,71.829152,12.449799,2023-08-27 00:00:00,16:54:00
146,147,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Gastón Vidal,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2023.Proy.SeguimientoAmbiental,Natales,13,"52°13'10,4''","72°52'26,1''",7.143333,7.166667,277.466667,0.012654,2.819486,2023-08-27 00:00:00,17:30:00


In [102]:
def limpieza_fisicoquimico(df):

    # Realizamos una copia del dataset
    df_fisico_copy = df.copy()

    # Eliminamos las filas innecesarias
    df_fisico_copy = df_fisico_copy.iloc[14: , :]

    # Dejamos la primera fila como columnas 
    df_fisico_copy.columns = df_fisico_copy.iloc[0]

    # Eliminamos la primera fila 
    df_fisico_copy = df_fisico_copy.iloc[1: , :]

    #Reiniciamos los indices
    df_fisico_copy = df_fisico_copy.reset_index(drop=True)

    #Seleccionamos las columnas
    df_fisico_copy = df_fisico_copy[['NOMB_PROY', 'ETAP_PROY', 'JEFE_PROY', 'INST_PROY', 'CAMPAÑA', 'ZONA',
       'ESTACION', 'Latitud S                ' , 'Longitud W ', 'pH' , 'Temperatura (°C)' , 'Redox EHNHE (mV)' , 'Sulfuros (µM)   ' , 'Materia orgánica total (%)',
        'Fecha',
       'Hora Muestreo' ]]

    # Agregamos la columna ID

    df_fisico_copy.insert(0 , 'ID' , df_fisico_copy.index + 1 )

    # Creamos una lista de columnas 

    lista_columnas = df_fisico_copy.columns.to_list()


    # Creamos una funcion para eliminar todo el contenido que esta en parentesis (*)

    def es_nan(x):
        return isinstance(x, float) and math.isnan(x)


    def quitar_parentesis_contenido(lista):
    
        resultado = []
        patron = re.compile(r'\([^)]*\)')  # todo lo que esté entre paréntesis (no anida)
        for c in lista:
            if es_nan(c) or c is None:
                resultado.append(c)
            elif isinstance(c, str):
                s = patron.sub('', c)                 # quita ( ... )
                s = re.sub(r'\s{2,}', ' ', s).strip() # colapsa espacios y recorta
                resultado.append(s)
            else:
                resultado.append(c)
        return resultado

    # Aplicamos la funcion para limpiar las columnas 
    lista_columnas = quitar_parentesis_contenido(lista_columnas)

    # Cambiamos las columnas del dataset

    df_fisico_copy.columns = lista_columnas

    # Creamos las columnas fijas y columnas variables

    indices_fijos = [0 , 1 , 2 ,3 ,4 ,5 ,6 ,7 , 8 , 9 ,  15 , 16]

    columnas_fijas = [lista_columnas[i] for i in indices_fijos]

    columnas_variables = [col for i, col in enumerate(lista_columnas) if i not in indices_fijos]

    
    # Ahora realizamos un melt para convertir las columnas variables en filas

    df_fisico_copy = pd.melt(df_fisico_copy , id_vars= columnas_fijas , value_vars= columnas_variables , var_name= 'VARIABLE' , value_name='VALOR' )

    # Convertimos la columna de VARIABLE a mayuscula

    df_fisico_copy['VARIABLE'] = df_fisico_copy['VARIABLE'].str.upper()

    #Corregimos los variables 's/d'
    df_fisico_copy['VALOR'] = np.where(df_fisico_copy['VALOR'] == 'SIN DATO', np.nan , df_fisico_copy['VALOR'] )
    df_fisico_copy['VALOR'] = np.where(df_fisico_copy['VALOR'] == 'sd', np.nan , df_fisico_copy['VALOR'] )
    df_fisico_copy['VALOR'] = np.where(df_fisico_copy['VALOR'] == 's/d', np.nan , df_fisico_copy['VALOR'] )
    
    #Eliminamos las filas con VALOR no nulo 

    df_fisico_copy = df_fisico_copy[df_fisico_copy['VALOR'].notnull()]

    #Reiniciamos los indices    
    df_fisico_copy = df_fisico_copy.reset_index(drop = True)

    # Cambiamos el tipo de dato de la columna Fecha 

    df_fisico_copy['Fecha'] = pd.to_datetime(df_fisico_copy['Fecha'], errors='coerce')

    #Creamos una columna para unir la Fecha y Hora Muestreo

    # Convertimos las Hora_Muestreo que son invalidos a 00:00:00
    df_fisico_copy['Hora Muestreo'] = np.where(df_fisico_copy['Hora Muestreo'] == 'sd' , datetime.time(0,0) , df_fisico_copy['Hora Muestreo'])

    #Creamos la columna llamada FECHA_MUESTREO que combine la fecha y la hora de muestreo
    df_fisico_copy['FECHA_MUESTREO'] = df_fisico_copy['Fecha'].dt.strftime('%d/%m/%Y') + " " + df_fisico_copy['Hora Muestreo'].astype(str)

    # Convertimos las variables 
    df_fisico_copy['VARIABLE'] = np.where(df_fisico_copy['VARIABLE'] == 'MATERIA ORGÁNICA TOTAL' , 'MATERIA ORGANICA' , df_fisico_copy['VARIABLE'])


    def normalizar(s):
            if pd.isna(s):
                return s
            # quitar tildes, bajar a minúsculas y recortar espacios
            s = ''.join(c for c in unicodedata.normalize('NFD', str(s).strip().lower())
                        if unicodedata.category(c) != 'Mn')
            return s

    # Agregamos la columna de COD_REGION
    
    # Diccionario en números romanos
        
    mapa_romano = {
            # Los Lagos (X)
            'reloncavi': 'X',
            'estuario': 'X',
            'calbuco': 'X',
            'maullin': 'X',
            'ancud': 'X',
            'chacao': 'X',
            'butachauques': 'X',
            'chaulinec': 'X',
            'tenaun': 'X',
            'chiloe central': 'X',
            'desertores': 'X',
            'quellon': 'X',
            'hornopiren': 'X',
            'ayacara': 'X',
            'chaiten': 'X',

            # Aysén (XI)
            'guaitecas': 'XI',
            'moraleda': 'XI',
            'puyuhuapi': 'XI',
            'aysen': 'XI',

            # Coquimbo (IV)
            'coquimbo': 'IV',
            'tongoy': 'IV',

            # Atacama (III)
            'atacama': 'III',

            # Magallanes (Natales) – XII
            'natales': 'XII',

            # Biobío (VIII)
            'bio-bio': 'VIII',

            # Los Ríos (XIV)
            'valdivia': 'XIV',
        }

    # Aplicamos la funcion para obtener el COD_REGION
    df_fisico_copy['COD_REGION'] = df_fisico_copy['ZONA'].apply(normalizar).map(mapa_romano)

    #Aplicamos una funcion para eliminar las ' y ° de LATITUD Y LONGITUD 

    df_fisico_copy['Longitud W'] = df_fisico_copy['Longitud W'].str.replace(r"[°'\"]", "", regex=True).str.rstrip().str.replace(',' , '.')
    df_fisico_copy['Latitud S'] = df_fisico_copy['Latitud S'].str.replace(r"[°'\"]", "", regex=True).str.rstrip().str.replace(',' , '.')

    #Renombramos los dataframes

    df_fisico_copy = df_fisico_copy.rename(columns={'Longitud W' : 'LONGITUD' , 'Latitud S' : 'LATITUD' , 'FECHA_MUESTREO' : 'FECHA'})

    #Filtramos las fechas que no sean nulas
    df_fisico_copy = df_fisico_copy[df_fisico_copy['FECHA'].notna()]



    return df_fisico_copy , lista_columnas 

In [103]:
df_fisico_copy ,  lista_columnas = limpieza_fisicoquimico(df_fisico)

In [106]:
df_fisico_copy.to_csv('output/2023/AMBIENTALES_FISICOQUIMICO_2023.csv' , index = False)